# 13_model_training_tcn_vscode_m2pro.ipynb — TCN local para Mac M2 Pro + VSCode

Versión adaptada para ejecutar fuera de Google Colab, usando un entorno local en macOS/Apple Silicon.

Objetivo:

```text
Usar secuencias temporales de las últimas 24 horas para predecir hs futura.
```

Horizontes:

```text
+3h
+6h
+12h
+24h
```

Cambios principales frente a la versión Colab:

- No monta Google Drive.
- Usa rutas locales compatibles con VSCode.
- Detecta Apple Silicon y GPU Metal si `tensorflow-metal` está instalado.
- Quita los límites de muestreo por defecto: entrena con todas las secuencias disponibles.
- Aumenta arquitectura, épocas y paciencia para un entrenamiento más serio.
- Evita construir el dataset de test antes del entrenamiento para liberar memoria.
- Guarda checkpoints, CSV de entrenamiento, métricas, predicciones y configuración reproducible.

Estructura esperada del proyecto:

```text
DeepWave Canarias/
├── gold/
│   └── training_dataset/
├── models/
└── ...
```

Si abres VSCode desde la carpeta `DeepWave Canarias`, el notebook detectará la ruta automáticamente. También puedes definir:

```bash
export DEEPWAVE_BASE_DIR="/ruta/a/DeepWave Canarias"
```


## Celda 0 — Entorno local recomendado para VSCode/macOS

En una terminal de VSCode, desde la raíz del proyecto:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip setuptools wheel
python -m pip install -r requirements-mac-m2pro.txt
python -m ipykernel install --user --name deepwave-m2pro --display-name "DeepWave M2 Pro"
```

Después selecciona el kernel **DeepWave M2 Pro** en VSCode.

Para comprobar que Metal está activo:

```bash
python - <<'PY'
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))
PY
```

Deberías ver una GPU tipo `PhysicalDevice(..., device_type='GPU')` si `tensorflow-metal` está funcionando.


In [ ]:
# Comprobación rápida del entorno de ejecución.
import platform
import sys
from pathlib import Path

print("Python:", sys.version)
print("Sistema:", platform.platform())
print("Máquina:", platform.machine())
print("Directorio actual:", Path.cwd())


: 

## Celda 1 — Dependencias

Este notebook no instala paquetes automáticamente para no mezclar entornos de VSCode. Usa el archivo `requirements-mac-m2pro.txt` incluido con esta versión.

Dependencias principales:

- `tensorflow`
- `tensorflow-metal`
- `pandas`
- `numpy`
- `pyarrow`
- `scikit-learn`
- `matplotlib`
- `joblib`
- `tqdm`
- `psutil`
- `ipykernel`


In [ ]:
# Verificación de dependencias críticas.
import importlib.util

required_modules = [
    "tensorflow",
    "pandas",
    "numpy",
    "pyarrow",
    "sklearn",
    "matplotlib",
    "joblib",
    "tqdm",
    "psutil",
]

missing = [m for m in required_modules if importlib.util.find_spec(m) is None]

if missing:
    raise ImportError(
        "Faltan dependencias: " + ", ".join(missing) +
        "\nInstala con: python -m pip install -r requirements-mac-m2pro.txt"
    )

print("Dependencias principales OK")


## Celda 2 — Imports, rutas locales y configuración Mac M2 Pro


In [ ]:
from pathlib import Path
import os
import sys
import platform
import json
import math
import gc
import warnings

# Ajustes antes de inicializar TensorFlow.
CPU_CORES = os.cpu_count() or 8
DEFAULT_INTRA_THREADS = max(1, CPU_CORES - 2)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("OMP_NUM_THREADS", str(DEFAULT_INTRA_THREADS))
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", str(DEFAULT_INTRA_THREADS))
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "2")

import pandas as pd
import numpy as np
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import joblib
import shutil
from tqdm.auto import tqdm

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

import matplotlib.pyplot as plt

try:
    import psutil
except Exception:
    psutil = None

warnings.filterwarnings("ignore")

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except Exception as e:
    raise ImportError(
        "No se pudo importar TensorFlow. En macOS/Apple Silicon instala el entorno con: "
        "python -m pip install -r requirements-mac-m2pro.txt"
    ) from e

# Threads TensorFlow. Puede fallar si el runtime ya fue inicializado; no es crítico.
try:
    tf.config.threading.set_intra_op_parallelism_threads(int(os.environ["TF_NUM_INTRAOP_THREADS"]))
    tf.config.threading.set_inter_op_parallelism_threads(int(os.environ["TF_NUM_INTEROP_THREADS"]))
except Exception as e:
    print("Aviso: no se pudieron fijar threads de TensorFlow:", repr(e))

# Evitar reserva agresiva si hay GPU disponible. En Metal puede no aplicar; se ignora si no está soportado.
try:
    gpus = tf.config.list_physical_devices("GPU")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
except Exception as e:
    print("Aviso: no se pudo consultar/configurar GPU:", repr(e))


def _env_int_or_none(name, default):
    value = os.getenv(name)
    if value is None:
        return default
    value = value.strip().lower()
    if value in {"", "none", "all", "full", "0"}:
        return None
    return int(value)


def _env_float(name, default):
    value = os.getenv(name)
    return default if value is None else float(value)


def _env_list_int(name, default):
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return list(default)
    return [int(x.strip()) for x in value.split(",") if x.strip()]


def resolve_base_dir():
    env_base = os.getenv("DEEPWAVE_BASE_DIR")
    candidates = []
    if env_base:
        candidates.append(Path(env_base).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path.home() / "AI Projects" / "DeepWave Canarias",
        Path.home() / "Documents" / "DeepWave Canarias",
        Path.home() / "DeepWave Canarias",
    ])

    seen = set()
    for base in candidates:
        base = base.expanduser().resolve()
        if base in seen:
            continue
        seen.add(base)
        if (base / "gold" / "training_dataset").exists():
            return base

    # Si no encuentra la ruta, devuelve cwd para que el error posterior indique qué falta.
    return Path(env_base).expanduser().resolve() if env_base else cwd


BASE_DIR = resolve_base_dir()
GOLD_DIR = BASE_DIR / "gold"
GOLD_TRAINING_DIR = GOLD_DIR / "training_dataset"

MODELS_DIR = BASE_DIR / "models" / "tcn"
RESULTS_DIR = GOLD_DIR / "model_results" / "tcn"
BACKUP_DIR = MODELS_DIR / "training_backup"

for d in [MODELS_DIR, RESULTS_DIR, BACKUP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HORIZONS_HOURS = [3, 6, 12, 24]
TARGET_COLS = [f"target_hs_{h}h" for h in HORIZONS_HOURS]
RISK_CLASSES = [0, 1, 2, 3]

RISK_THRESHOLDS = {
    "low_max": 1.0,
    "moderate_max": 2.0,
    "high_max": 3.0,
}

# -------------------------------------------------------------------------
# PERFIL LOCAL PARA MAC M2 PRO
# -------------------------------------------------------------------------
# Por defecto usa todas las secuencias disponibles. Puedes cambiar perfil con:
# export DEEPWAVE_TRAIN_PROFILE=m2pro_balanced
# o sobreescribir variables concretas, por ejemplo:
# export DEEPWAVE_BATCH_SIZE=256
# export DEEPWAVE_EPOCHS=100
# export DEEPWAVE_MAX_TRAIN_SEQUENCES=300000

TRAIN_PROFILE = os.getenv("DEEPWAVE_TRAIN_PROFILE", "m2pro_full").lower()

TRAIN_PROFILES = {
    "m2pro_full": {
        "batch_size": 512,
        "epochs": 150,
        "patience": 18,
        "max_train_sequences": None,
        "max_val_sequences": None,
        "max_test_sequences": None,
        "max_tcn_features": 96,
        "tcn_filters": 96,
        "tcn_dilations": [1, 2, 4, 8, 16, 32],
        "tcn_dropout": 0.15,
        "dense_units_1": 192,
        "dense_units_2": 96,
        "learning_rate": 3e-4,
    },
    "m2pro_balanced": {
        "batch_size": 256,
        "epochs": 90,
        "patience": 12,
        "max_train_sequences": 350000,
        "max_val_sequences": 90000,
        "max_test_sequences": 120000,
        "max_tcn_features": 80,
        "tcn_filters": 64,
        "tcn_dilations": [1, 2, 4, 8, 16],
        "tcn_dropout": 0.12,
        "dense_units_1": 128,
        "dense_units_2": 64,
        "learning_rate": 5e-4,
    },
    "debug": {
        "batch_size": 128,
        "epochs": 3,
        "patience": 2,
        "max_train_sequences": 10000,
        "max_val_sequences": 3000,
        "max_test_sequences": 3000,
        "max_tcn_features": 48,
        "tcn_filters": 32,
        "tcn_dilations": [1, 2, 4],
        "tcn_dropout": 0.10,
        "dense_units_1": 64,
        "dense_units_2": 32,
        "learning_rate": 1e-3,
    },
}

if TRAIN_PROFILE not in TRAIN_PROFILES:
    raise ValueError(f"TRAIN_PROFILE desconocido: {TRAIN_PROFILE}. Opciones: {list(TRAIN_PROFILES)}")

PROFILE = TRAIN_PROFILES[TRAIN_PROFILE]

SEQUENCE_LENGTH = _env_int_or_none("DEEPWAVE_SEQUENCE_LENGTH", 24) or 24
BATCH_SIZE = _env_int_or_none("DEEPWAVE_BATCH_SIZE", PROFILE["batch_size"]) or PROFILE["batch_size"]
EPOCHS = _env_int_or_none("DEEPWAVE_EPOCHS", PROFILE["epochs"]) or PROFILE["epochs"]
PATIENCE = _env_int_or_none("DEEPWAVE_PATIENCE", PROFILE["patience"]) or PROFILE["patience"]
RANDOM_STATE = _env_int_or_none("DEEPWAVE_RANDOM_STATE", 42) or 42

USE_COMMON_ZONES_ONLY = os.getenv("DEEPWAVE_USE_COMMON_ZONES_ONLY", "1").strip() not in {"0", "false", "False"}

MAX_TRAIN_SEQUENCES = _env_int_or_none("DEEPWAVE_MAX_TRAIN_SEQUENCES", PROFILE["max_train_sequences"])
MAX_VAL_SEQUENCES = _env_int_or_none("DEEPWAVE_MAX_VAL_SEQUENCES", PROFILE["max_val_sequences"])
MAX_TEST_SEQUENCES = _env_int_or_none("DEEPWAVE_MAX_TEST_SEQUENCES", PROFILE["max_test_sequences"])
MAX_TCN_FEATURES = _env_int_or_none("DEEPWAVE_MAX_TCN_FEATURES", PROFILE["max_tcn_features"])

TCN_FILTERS = _env_int_or_none("DEEPWAVE_TCN_FILTERS", PROFILE["tcn_filters"]) or PROFILE["tcn_filters"]
TCN_DILATIONS = _env_list_int("DEEPWAVE_TCN_DILATIONS", PROFILE["tcn_dilations"])
TCN_DROPOUT = _env_float("DEEPWAVE_TCN_DROPOUT", PROFILE["tcn_dropout"])
DENSE_UNITS_1 = _env_int_or_none("DEEPWAVE_DENSE_UNITS_1", PROFILE["dense_units_1"]) or PROFILE["dense_units_1"]
DENSE_UNITS_2 = _env_int_or_none("DEEPWAVE_DENSE_UNITS_2", PROFILE["dense_units_2"]) or PROFILE["dense_units_2"]
LEARNING_RATE = _env_float("DEEPWAVE_LEARNING_RATE", PROFILE["learning_rate"])
HUBER_DELTA = _env_float("DEEPWAVE_HUBER_DELTA", 0.5)
WEIGHT_DECAY = _env_float("DEEPWAVE_WEIGHT_DECAY", 1e-4)

# No guardar metadata de train por defecto: ahorra RAM y disco. Val/test sí se guardan para métricas.
SAVE_TRAIN_METADATA = os.getenv("DEEPWAVE_SAVE_TRAIN_METADATA", "0").strip() in {"1", "true", "True"}

# Features con demasiados nulos se excluyen.
MAX_MISSING_PCT_FEATURE = _env_float("DEEPWAVE_MAX_MISSING_PCT_FEATURE", 98.0)

# No usar zona_id como feature para evitar memorizar identificadores.
USE_ZONA_ID_AS_FEATURE = os.getenv("DEEPWAVE_USE_ZONA_ID_AS_FEATURE", "0").strip() in {"1", "true", "True"}

PREFERRED_SEQUENCE_FEATURES = [
    "simar_hs",
    "simar_tp",
    "simar_tm02",
    "simar_wave_direction",
    "simar_swell_height",
    "simar_swell_period",
    "simar_swell_direction",
    "simar_wind_wave_height",
    "simar_wind_speed",
    "simar_wind_direction",
    "simar_u10",
    "simar_v10",
    "simar_current_speed",
    "simar_current_direction",
    "simar_sst",
    "simar_salinity",
    "redmar_sea_level",
    "redmar_astronomical_tide",
    "redmar_meteorological_residual",
    "redmar_daily_tidal_range",
    "redmar_hours_to_high_tide",
    "redmar_hours_to_low_tide",
    "aemet_temperature_air",
    "aemet_precipitation",
    "aemet_wind_speed",
    "aemet_pressure",
    "lat",
    "lon",
    "depth_100m",
    "depth_500m",
    "depth_1km",
    "depth_2km",
    "mean_depth_1km",
    "slope_0_500m",
    "slope_500m_2km",
    "distance_to_10m_isobath",
    "distance_to_20m_isobath",
    "bathymetry_roughness",
    "exposicion_norte",
    "exposicion_oeste",
    "exposicion_este",
    "exposicion_swell_nw",
    "exposicion_swell_ne",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "dayofyear_sin",
    "dayofyear_cos",
]

CATEGORICAL_CANDIDATES = [
    "isla",
    "orientacion_costa",
    "redmar_tide_phase",
]

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

if psutil is not None:
    vm = psutil.virtual_memory()
    RAM_TOTAL_GB = vm.total / 1024**3
    RAM_AVAILABLE_GB = vm.available / 1024**3
else:
    RAM_TOTAL_GB = None
    RAM_AVAILABLE_GB = None

print("TensorFlow:", tf.__version__)
print("Apple Silicon:", platform.system() == "Darwin" and platform.machine() == "arm64")
print("GPU TensorFlow:", tf.config.list_physical_devices("GPU"))
print("CPU cores:", CPU_CORES)
print("TF intra/inter threads:", os.environ.get("TF_NUM_INTRAOP_THREADS"), os.environ.get("TF_NUM_INTEROP_THREADS"))
if RAM_TOTAL_GB is not None:
    print(f"RAM total≈{RAM_TOTAL_GB:.1f} GB | disponible≈{RAM_AVAILABLE_GB:.1f} GB")

print("BASE_DIR:", BASE_DIR)
print("GOLD_TRAINING_DIR:", GOLD_TRAINING_DIR)
print("Existe training dataset:", GOLD_TRAINING_DIR.exists())
print("Perfil:", TRAIN_PROFILE)
print("BATCH_SIZE:", BATCH_SIZE)
print("EPOCHS/PATIENCE:", EPOCHS, PATIENCE)
print("MAX_TRAIN/VAL/TEST_SEQUENCES:", MAX_TRAIN_SEQUENCES, MAX_VAL_SEQUENCES, MAX_TEST_SEQUENCES)
print("MAX_TCN_FEATURES:", MAX_TCN_FEATURES)
print("TCN_FILTERS:", TCN_FILTERS)
print("TCN_DILATIONS:", TCN_DILATIONS)

if not GOLD_TRAINING_DIR.exists():
    raise FileNotFoundError(
        "No existe gold/training_dataset/. Abre VSCode desde la raíz del proyecto o define "
        "DEEPWAVE_BASE_DIR=/ruta/a/DeepWave Canarias. Ejecuta primero 11_gold_training_dataset.ipynb."
    )


## Celda 3 — Funciones auxiliares

In [4]:
def ensure_utc(series):
    return pd.to_datetime(series, utc=True, errors="coerce")


def risk_level_from_hs(values):
    values = pd.Series(np.asarray(values), dtype="float64")
    risk = pd.Series(np.nan, index=values.index, dtype="float64")

    risk.loc[values < RISK_THRESHOLDS["low_max"]] = 0
    risk.loc[(values >= RISK_THRESHOLDS["low_max"]) & (values < RISK_THRESHOLDS["moderate_max"])] = 1
    risk.loc[(values >= RISK_THRESHOLDS["moderate_max"]) & (values < RISK_THRESHOLDS["high_max"])] = 2
    risk.loc[values >= RISK_THRESHOLDS["high_max"]] = 3

    return risk.astype("Int64")


def regression_metrics(y_true, y_pred):
    y_true = pd.Series(np.asarray(y_true), dtype="float64")
    y_pred = pd.Series(np.asarray(y_pred), dtype="float64")

    mask = y_true.notna() & y_pred.notna()

    if mask.sum() == 0:
        return {
            "n": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "bias": np.nan,
            "corr": np.nan,
        }

    yt = y_true[mask].values
    yp = y_pred[mask].values

    return {
        "n": int(mask.sum()),
        "mae": float(mean_absolute_error(yt, yp)),
        "rmse": float(np.sqrt(mean_squared_error(yt, yp))),
        "r2": float(r2_score(yt, yp)) if len(np.unique(yt)) > 1 else np.nan,
        "bias": float(np.mean(yp - yt)),
        "corr": float(np.corrcoef(yt, yp)[0, 1]) if len(yt) > 1 and np.std(yt) > 0 and np.std(yp) > 0 else np.nan,
    }


def classification_metrics(y_true, y_pred):
    y_true = pd.Series(np.asarray(y_true), dtype="Int64")
    y_pred = pd.Series(np.asarray(y_pred), dtype="Int64")

    mask = y_true.notna() & y_pred.notna()

    if mask.sum() == 0:
        out = {
            "n": 0,
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "macro_f1": np.nan,
            "weighted_f1": np.nan,
        }

        for cls in RISK_CLASSES:
            out[f"precision_class_{cls}"] = np.nan
            out[f"recall_class_{cls}"] = np.nan
            out[f"f1_class_{cls}"] = np.nan
            out[f"support_class_{cls}"] = 0

        return out

    yt = y_true[mask].astype(int).values
    yp = y_pred[mask].astype(int).values

    precision, recall, f1, support = precision_recall_fscore_support(
        yt,
        yp,
        labels=RISK_CLASSES,
        zero_division=0,
    )

    out = {
        "n": int(mask.sum()),
        "accuracy": float(accuracy_score(yt, yp)),
        "balanced_accuracy": float(balanced_accuracy_score(yt, yp)),
        "macro_f1": float(f1_score(yt, yp, labels=RISK_CLASSES, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(yt, yp, labels=RISK_CLASSES, average="weighted", zero_division=0)),
    }

    for i, cls in enumerate(RISK_CLASSES):
        out[f"precision_class_{cls}"] = float(precision[i])
        out[f"recall_class_{cls}"] = float(recall[i])
        out[f"f1_class_{cls}"] = float(f1[i])
        out[f"support_class_{cls}"] = int(support[i])

    return out


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def memory_report(name, df):
    mb = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"{name}: shape={df.shape}, memoria≈{mb:.1f} MB")


def sample_indices(n, max_n):
    if max_n is None or n <= max_n:
        return np.arange(n)
    rng = np.random.default_rng(RANDOM_STATE)
    return np.sort(rng.choice(np.arange(n), size=max_n, replace=False))


def plot_history(history):
    hist = pd.DataFrame(history.history)

    plt.figure(figsize=(9, 5))
    plt.plot(hist["loss"], label="train loss")
    if "val_loss" in hist.columns:
        plt.plot(hist["val_loss"], label="val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Curva de entrenamiento TCN")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "plot_training_history_loss.png", dpi=160)
    plt.show()

    if "mae" in hist.columns:
        plt.figure(figsize=(9, 5))
        plt.plot(hist["mae"], label="train mae")
        if "val_mae" in hist.columns:
            plt.plot(hist["val_mae"], label="val mae")
        plt.xlabel("Epoch")
        plt.ylabel("MAE")
        plt.title("Curva MAE TCN")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "plot_training_history_mae.png", dpi=160)
        plt.show()

## Celda 4 — Cargar Gold training dataset

In [ ]:
dataset = ds.dataset(str(GOLD_TRAINING_DIR), format="parquet", partitioning="hive")

parquet_files = list(GOLD_TRAINING_DIR.rglob("*.parquet"))
print("Parquet files:", len(parquet_files))

# En Mac local usamos lectura multihilo de Arrow. El generador posterior evita materializar
# todas las ventanas temporales en RAM.
gold = dataset.to_table(use_threads=True).to_pandas()

gold["timestamp"] = ensure_utc(gold["timestamp"])
gold["split"] = gold["split"].astype(str)

print("Gold cargado:")
memory_report("gold", gold)

print("Splits:")
display(gold["split"].value_counts(dropna=False).reset_index())

print("Rango temporal:")
print(gold["timestamp"].min(), "→", gold["timestamp"].max())

print("Zonas:", gold["zona_id"].nunique())
display(gold.head())


## Celda 5 — Filtrar zonas comunes train/val/test

In [ ]:
if USE_COMMON_ZONES_ONLY:
    zones_by_split = {
        split: set(gold.loc[gold["split"] == split, "zona_id"].dropna().astype(str).unique())
        for split in ["train", "val", "test"]
    }

    common_zones = set.intersection(*zones_by_split.values())

    print("Zonas train:", len(zones_by_split["train"]))
    print("Zonas val:", len(zones_by_split["val"]))
    print("Zonas test:", len(zones_by_split["test"]))
    print("Zonas comunes:", len(common_zones))

    if len(common_zones) == 0:
        raise ValueError("No hay zonas comunes entre train/val/test.")

    before = len(gold)
    gold = gold[gold["zona_id"].astype(str).isin(common_zones)].copy()
    after = len(gold)

    print(f"Filtrado a zonas comunes: {before:,} → {after:,} filas")

else:
    common_zones = set(gold["zona_id"].dropna().astype(str).unique())
    print("Se usan todas las zonas:", len(common_zones))

split_zone_summary = (
    gold.groupby("split", as_index=False)
    .agg(
        rows=("zona_id", "size"),
        zones=("zona_id", "nunique"),
        timestamp_min=("timestamp", "min"),
        timestamp_max=("timestamp", "max"),
    )
)

display(split_zone_summary)
split_zone_summary.to_csv(RESULTS_DIR / "split_zone_summary.csv", index=False)

## Celda 6 — Selección de features secuenciales

In [ ]:
hard_exclude = set([
    "timestamp",
    "date",
    "split",
    "gold_dataset_version",
    "target_source",
    "has_any_target",
    "is_trainable_target_3h",
    "risk_threshold_low_max",
    "risk_threshold_moderate_max",
    "risk_threshold_high_max",
    "nombre_zona",
    "municipio",
])

if not USE_ZONA_ID_AS_FEATURE:
    hard_exclude.add("zona_id")

for c in gold.columns:
    if c.startswith("target_"):
        hard_exclude.add(c)

# Evitar usar lags explícitos en el TCN para no duplicar la lógica temporal.
# El TCN aprende la memoria desde la secuencia de las últimas 24h.
for c in gold.columns:
    if "_lag_" in c or "_roll_" in c:
        hard_exclude.add(c)

feature_audit_rows = []

for c in gold.columns:
    missing_pct = float(gold[c].isna().mean() * 100)
    nunique = int(gold[c].nunique(dropna=True))

    use = True
    reason = ""

    if c in hard_exclude:
        use = False
        reason = "hard_exclude"
    elif missing_pct >= MAX_MISSING_PCT_FEATURE:
        use = False
        reason = f"missing_pct_ge_{MAX_MISSING_PCT_FEATURE}"
    elif nunique <= 1:
        use = False
        reason = "constant_or_all_missing"
    elif str(gold[c].dtype).startswith("datetime"):
        use = False
        reason = "datetime"

    feature_audit_rows.append(
        {
            "column": c,
            "dtype": str(gold[c].dtype),
            "missing_pct": missing_pct,
            "nunique": nunique,
            "selected_candidate": use,
            "reason_excluded": reason,
        }
    )

feature_audit = pd.DataFrame(feature_audit_rows)

valid_candidates = set(feature_audit.loc[feature_audit["selected_candidate"], "column"])

preferred_existing = [
    c for c in PREFERRED_SEQUENCE_FEATURES
    if c in valid_candidates and c in gold.columns
]

extra = [
    c for c in feature_audit.loc[feature_audit["selected_candidate"], "column"].tolist()
    if c not in preferred_existing and c not in CATEGORICAL_CANDIDATES
]

all_sequence_numeric_features = preferred_existing + extra

if MAX_TCN_FEATURES is None:
    sequence_numeric_features = all_sequence_numeric_features
else:
    sequence_numeric_features = all_sequence_numeric_features[:MAX_TCN_FEATURES]

categorical_features = [
    c for c in CATEGORICAL_CANDIDATES
    if c in valid_candidates and c in gold.columns
]

if len(sequence_numeric_features) == 0:
    raise ValueError("No se seleccionó ninguna feature numérica para el TCN.")

print("Features numéricas/secuenciales:", len(sequence_numeric_features))
print(sequence_numeric_features)

print("Categóricas codificadas:", categorical_features)
print("Features candidatas totales antes de cap:", len(all_sequence_numeric_features))

feature_audit["selected_for_tcn"] = feature_audit["column"].isin(sequence_numeric_features + categorical_features)

display(feature_audit.sort_values(["selected_for_tcn", "selected_candidate", "missing_pct"], ascending=[False, False, False]).head(100))

feature_audit.to_csv(RESULTS_DIR / "feature_selection_audit_tcn.csv", index=False)


## Celda 7 — Codificar categóricas e imputar/escalar features

In [ ]:
# Codificar categóricas con mapping aprendido en train.
category_maps = {}

encoded_categorical_features = []

for c in categorical_features:
    train_values = (
        gold.loc[gold["split"] == "train", c]
        .astype("string")
        .fillna("UNKNOWN")
        .unique()
        .tolist()
    )

    train_values = sorted([str(v) for v in train_values])
    mapping = {v: i for i, v in enumerate(train_values)}

    category_maps[c] = mapping

    encoded_col = c + "_encoded"
    gold[encoded_col] = (
        gold[c]
        .astype("string")
        .fillna("UNKNOWN")
        .map(mapping)
        .fillna(-1)
        .astype("float32")
    )

    encoded_categorical_features.append(encoded_col)

# Features finales para el TCN.
feature_cols = sequence_numeric_features + encoded_categorical_features

# Convertir a numérico.
for c in feature_cols:
    gold[c] = pd.to_numeric(gold[c], errors="coerce").astype("float32")

# Imputación y escala con train.
train_mask = gold["split"] == "train"

feature_medians = {}
feature_means = {}
feature_stds = {}

for c in tqdm(feature_cols, desc="Imputando/escalando"):
    median = float(gold.loc[train_mask, c].median()) if gold.loc[train_mask, c].notna().any() else 0.0

    gold[c] = gold[c].fillna(median)

    mean = float(gold.loc[train_mask, c].mean())
    std = float(gold.loc[train_mask, c].std())

    if not np.isfinite(std) or std == 0:
        std = 1.0

    gold[c] = ((gold[c] - mean) / std).astype("float32")

    feature_medians[c] = median
    feature_means[c] = mean
    feature_stds[c] = std

# Targets.
for c in TARGET_COLS:
    gold[c] = pd.to_numeric(gold[c], errors="coerce").astype("float32")

# Persistencia actual.
gold["persistence_hs"] = pd.to_numeric(gold["simar_hs"], errors="coerce").astype("float32")

scaler_metadata = {
    "feature_cols": feature_cols,
    "sequence_numeric_features": sequence_numeric_features,
    "categorical_features": categorical_features,
    "encoded_categorical_features": encoded_categorical_features,
    "category_maps": category_maps,
    "feature_medians": feature_medians,
    "feature_means": feature_means,
    "feature_stds": feature_stds,
    "sequence_length": SEQUENCE_LENGTH,
    "horizons_hours": HORIZONS_HOURS,
}

save_json(scaler_metadata, RESULTS_DIR / "tcn_feature_scaler_metadata.json")
save_json(scaler_metadata, MODELS_DIR / "tcn_feature_scaler_metadata.json")

memory_report("gold features preparadas", gold[feature_cols + TARGET_COLS + ["split", "timestamp", "zona_id"]])
display(gold[feature_cols].head())

## Celda 8 — Crear datasets secuenciales `tf.data`

In [ ]:
# Celda 8 — Crear datasets secuenciales RAM-safe para entrenamiento local.
#
# Se mantienen los datos base por zona y se generan ventanas al vuelo por batch.
# Diferencia importante frente a Colab: por defecto no se muestrea train/val/test.
# Además, el dataset de test se construye después del entrenamiento para liberar RAM durante model.fit.

class MacTCNSequence(keras.utils.Sequence):
    def __init__(
        self,
        features_by_group,
        targets_by_group,
        sample_group_ids,
        sample_end_idxs,
        batch_size,
        shuffle=False,
        random_state=RANDOM_STATE,
        name="dataset",
    ):
        super().__init__()
        self.features_by_group = features_by_group
        self.targets_by_group = targets_by_group
        self.sample_group_ids = np.asarray(sample_group_ids, dtype=np.int32)
        self.sample_end_idxs = np.asarray(sample_end_idxs, dtype=np.int32)
        self.batch_size = int(batch_size)
        self.shuffle = bool(shuffle)
        self.random_state = int(random_state)
        self.name = name
        self.rng = np.random.default_rng(self.random_state)
        self.order = np.arange(len(self.sample_group_ids), dtype=np.int32)
        self.on_epoch_end()

    @property
    def n_samples(self):
        return int(len(self.sample_group_ids))

    def __len__(self):
        return int(np.ceil(len(self.order) / self.batch_size))

    def __getitem__(self, batch_idx):
        start = batch_idx * self.batch_size
        end = min((batch_idx + 1) * self.batch_size, len(self.order))
        batch_order = self.order[start:end]

        batch_group_ids = self.sample_group_ids[batch_order]
        batch_end_idxs = self.sample_end_idxs[batch_order]

        X = np.empty(
            (len(batch_order), SEQUENCE_LENGTH, len(feature_cols)),
            dtype=np.float32,
        )
        y = np.empty(
            (len(batch_order), len(TARGET_COLS)),
            dtype=np.float32,
        )

        # Bucle pequeño por batch. Evita materializar todas las ventanas en RAM.
        for j, (g, end_idx) in enumerate(zip(batch_group_ids, batch_end_idxs)):
            start_idx = end_idx - SEQUENCE_LENGTH + 1
            X[j] = self.features_by_group[g][start_idx:end_idx + 1]
            y[j] = self.targets_by_group[g][end_idx]

        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.order)


def contiguous_valid_end_indices(timestamps, target_valid, sequence_length):
    """
    Devuelve end_idx válidos:
    - end_idx >= sequence_length - 1
    - todos los targets de la fila final son válidos
    - las 24 horas de la ventana son consecutivas aproximadamente.
    """
    n = len(timestamps)

    if n < sequence_length:
        return np.array([], dtype=np.int32)

    ts = pd.to_datetime(timestamps, utc=True, errors="coerce")
    diffs = pd.Series(ts).diff().dt.total_seconds().to_numpy()

    bad_gap = np.zeros(n, dtype=np.int32)
    bad_gap[1:] = (~np.isclose(diffs[1:], 3600.0, atol=5.0)).astype(np.int32)

    cum_bad = np.cumsum(bad_gap)

    all_end = np.arange(sequence_length - 1, n, dtype=np.int32)
    start = all_end - sequence_length + 1

    bad_count = cum_bad[all_end] - cum_bad[start]
    valid = (bad_count == 0) & target_valid[all_end]

    return all_end[valid].astype(np.int32)


def build_split_sequence(split_name, max_sequences_total=None, shuffle=False, build_metadata=True):
    split_df = gold.loc[gold["split"] == split_name].sort_values(["zona_id", "timestamp"]).reset_index(drop=True)
    groups = list(split_df.groupby("zona_id", sort=True))

    features_by_group = []
    targets_by_group = []
    meta_by_group = []
    sample_group_ids = []
    sample_end_idxs = []

    for _, (zona_id, g) in enumerate(tqdm(groups, desc=f"Preparando grupos {split_name}")):
        g = g.sort_values("timestamp").reset_index(drop=True)

        target_valid = g[TARGET_COLS].notna().all(axis=1).to_numpy()

        end_indices = contiguous_valid_end_indices(
            g["timestamp"],
            target_valid=target_valid,
            sequence_length=SEQUENCE_LENGTH,
        )

        if len(end_indices) == 0:
            continue

        X_group = g[feature_cols].to_numpy(dtype=np.float32, copy=True)
        y_group = g[TARGET_COLS].to_numpy(dtype=np.float32, copy=True)

        features_by_group.append(X_group)
        targets_by_group.append(y_group)

        real_group_idx = len(features_by_group) - 1
        sample_group_ids.append(np.full(len(end_indices), real_group_idx, dtype=np.int32))
        sample_end_idxs.append(end_indices.astype(np.int32))

        if build_metadata:
            meta_cols = [
                "zona_id",
                "timestamp",
                "split",
                "isla",
                "lat",
                "lon",
                "simar_hs",
                "persistence_hs",
            ] + TARGET_COLS
            meta_cols = [c for c in meta_cols if c in g.columns]
            meta_by_group.append(g.iloc[end_indices][meta_cols].copy())

    if not sample_group_ids:
        return None, pd.DataFrame()

    sample_group_ids = np.concatenate(sample_group_ids)
    sample_end_idxs = np.concatenate(sample_end_idxs)

    n_total = len(sample_group_ids)
    print(f"{split_name}: secuencias candidatas antes de muestreo:", f"{n_total:,}")

    if max_sequences_total is not None and n_total > max_sequences_total:
        rng = np.random.default_rng(RANDOM_STATE)
        chosen = np.sort(rng.choice(np.arange(n_total), size=max_sequences_total, replace=False))
        sample_group_ids = sample_group_ids[chosen]
        sample_end_idxs = sample_end_idxs[chosen]

        if build_metadata:
            full_meta = pd.concat(meta_by_group, ignore_index=True)
            meta = full_meta.iloc[chosen].reset_index(drop=True)
            del full_meta
        else:
            meta = pd.DataFrame()

        print(f"{split_name}: secuencias tras muestreo:", f"{len(sample_group_ids):,}")
    else:
        meta = pd.concat(meta_by_group, ignore_index=True) if build_metadata else pd.DataFrame()

    seq = MacTCNSequence(
        features_by_group=features_by_group,
        targets_by_group=targets_by_group,
        sample_group_ids=sample_group_ids,
        sample_end_idxs=sample_end_idxs,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        random_state=RANDOM_STATE,
        name=split_name,
    )

    return seq, meta


metadata_base_cols = ["zona_id", "timestamp", "split", "isla", "lat", "lon", "simar_hs", "persistence_hs"]
missing_required = [c for c in TARGET_COLS + ["zona_id", "timestamp", "split"] if c not in gold.columns]
if missing_required:
    raise ValueError("Faltan columnas requeridas: " + ", ".join(missing_required))

needed_cols = list(dict.fromkeys(
    [c for c in feature_cols if c in gold.columns]
    + TARGET_COLS
    + [c for c in metadata_base_cols if c in gold.columns]
))

gold = gold[needed_cols].copy()
gc.collect()

train_ds, train_meta = build_split_sequence(
    "train",
    max_sequences_total=MAX_TRAIN_SEQUENCES,
    shuffle=True,
    build_metadata=SAVE_TRAIN_METADATA,
)

val_ds, val_meta = build_split_sequence(
    "val",
    max_sequences_total=MAX_VAL_SEQUENCES,
    shuffle=False,
    build_metadata=True,
)

if train_ds is None or val_ds is None:
    raise ValueError("Train o val está vacío.")

TRAIN_SEQUENCE_COUNT = train_ds.n_samples
VAL_SEQUENCE_COUNT = val_ds.n_samples

print("Batches train:", len(train_ds))
print("Batches val:", len(val_ds))
print("Secuencias train:", f"{TRAIN_SEQUENCE_COUNT:,}")
print("Secuencias val:", f"{VAL_SEQUENCE_COUNT:,}")

if not train_meta.empty:
    display(train_meta.head())
    train_meta.to_csv(RESULTS_DIR / "train_sequence_metadata.csv", index=False)
else:
    print("train_meta no se guarda para ahorrar RAM. Activa DEEPWAVE_SAVE_TRAIN_METADATA=1 si la necesitas.")

display(val_meta.head())
val_meta.to_csv(RESULTS_DIR / "val_sequence_metadata.csv", index=False)

gc.collect()


# Celda 8B — Comprobación de variables necesarias antes de crear el TCN


In [ ]:
def residual_tcn_block(x, filters, kernel_size, dilation_rate, dropout_rate, name):
    shortcut = x

    y = layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation=None,
        kernel_regularizer=keras.regularizers.l2(WEIGHT_DECAY),
        name=f"{name}_conv1",
    )(x)
    y = layers.LayerNormalization(name=f"{name}_ln1")(y)
    y = layers.Activation("gelu", name=f"{name}_gelu1")(y)
    y = layers.SpatialDropout1D(dropout_rate, name=f"{name}_spdrop1")(y)

    y = layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation=None,
        kernel_regularizer=keras.regularizers.l2(WEIGHT_DECAY),
        name=f"{name}_conv2",
    )(y)
    y = layers.LayerNormalization(name=f"{name}_ln2")(y)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(
            filters=filters,
            kernel_size=1,
            padding="same",
            kernel_regularizer=keras.regularizers.l2(WEIGHT_DECAY),
            name=f"{name}_shortcut",
        )(shortcut)

    out = layers.Add(name=f"{name}_add")([shortcut, y])
    out = layers.Activation("gelu", name=f"{name}_gelu_out")(out)

    return out


def build_tcn_model(sequence_length, n_features, n_outputs):
    inputs = keras.Input(shape=(sequence_length, n_features), name="sequence_input")

    x = layers.Conv1D(
        TCN_FILTERS,
        kernel_size=1,
        padding="same",
        activation="gelu",
        kernel_regularizer=keras.regularizers.l2(WEIGHT_DECAY),
        name="input_projection",
    )(inputs)

    for i, d in enumerate(TCN_DILATIONS):
        x = residual_tcn_block(
            x,
            filters=TCN_FILTERS,
            kernel_size=3,
            dilation_rate=d,
            dropout_rate=TCN_DROPOUT,
            name=f"tcn_block_{i+1}_d{d}",
        )

    x_avg = layers.GlobalAveragePooling1D(name="global_avg_pool")(x)
    x_max = layers.GlobalMaxPooling1D(name="global_max_pool")(x)
    x = layers.Concatenate(name="temporal_pool_concat")([x_avg, x_max])

    x = layers.Dense(
        DENSE_UNITS_1,
        activation="gelu",
        kernel_regularizer=keras.regularizers.l2(WEIGHT_DECAY),
        name=f"dense_{DENSE_UNITS_1}",
    )(x)
    x = layers.Dropout(0.25, name="dense_dropout_1")(x)
    x = layers.Dense(
        DENSE_UNITS_2,
        activation="gelu",
        kernel_regularizer=keras.regularizers.l2(WEIGHT_DECAY),
        name=f"dense_{DENSE_UNITS_2}",
    )(x)
    x = layers.Dropout(0.10, name="dense_dropout_2")(x)

    outputs = layers.Dense(n_outputs, activation="linear", name="hs_outputs")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="DeepWave_TCN_M2Pro_multi_horizon")

    try:
        optimizer = keras.optimizers.AdamW(
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            clipnorm=1.0,
        )
    except Exception:
        optimizer = keras.optimizers.Adam(
            learning_rate=LEARNING_RATE,
            clipnorm=1.0,
        )

    model.compile(
        optimizer=optimizer,
        loss=keras.losses.Huber(delta=HUBER_DELTA),
        metrics=[
            keras.metrics.MeanAbsoluteError(name="mae"),
            keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )

    return model


n_features = len(feature_cols)
n_outputs = len(TARGET_COLS)

model = build_tcn_model(SEQUENCE_LENGTH, n_features, n_outputs)
model.summary()

save_json(
    {
        "sequence_length": SEQUENCE_LENGTH,
        "n_features": n_features,
        "n_outputs": n_outputs,
        "feature_cols": feature_cols,
        "target_cols": TARGET_COLS,
        "architecture": "Residual TCN optimized for local Apple Silicon training",
        "train_profile": TRAIN_PROFILE,
        "tcn_filters": TCN_FILTERS,
        "dilations": TCN_DILATIONS,
        "dropout": TCN_DROPOUT,
        "dense_units_1": DENSE_UNITS_1,
        "dense_units_2": DENSE_UNITS_2,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "huber_delta": HUBER_DELTA,
    },
    RESULTS_DIR / "tcn_model_architecture.json",
)


# Celda 9 — Definir arquitectura TCN optimizada para Mac M2 Pro


In [ ]:
callbacks = [
    keras.callbacks.CSVLogger(
        filename=str(RESULTS_DIR / "training_log.csv"),
        append=False,
    ),
    keras.callbacks.BackupAndRestore(
        backup_dir=str(BACKUP_DIR),
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=max(3, PATIENCE // 3),
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODELS_DIR / "tcn_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.TerminateOnNaN(),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

# Guardar modelo final.
model.save(MODELS_DIR / "tcn_final.keras")

# Guardar history.
history_df = pd.DataFrame(history.history)
history_df.to_csv(RESULTS_DIR / "training_history.csv", index=False)

plot_history(history)

print("Modelo guardado:")
print(MODELS_DIR / "tcn_best.keras")
print(MODELS_DIR / "tcn_final.keras")

# Liberar train antes de construir test/predicciones.
del train_ds
gc.collect()


gc.collect()

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=max(2, PATIENCE // 2),
        min_lr=1e-5,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODELS_DIR / "tcn_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

# Guardar modelo final.
model.save(MODELS_DIR / "tcn_final.keras")

# Guardar history.
history_df = pd.DataFrame(history.history)
history_df.to_csv(RESULTS_DIR / "training_history.csv", index=False)

plot_history(history)

print("Modelo guardado:")
print(MODELS_DIR / "tcn_best.keras")
print(MODELS_DIR / "tcn_final.keras")

In [ ]:
def predict_dataset(ds_split, meta_split, split_name):
    pred = model.predict(ds_split, verbose=1)
    pred = np.clip(pred, 0, None)

    out = meta_split.copy().reset_index(drop=True)

    if len(out) != pred.shape[0]:
        raise ValueError(f"Desalineación {split_name}: meta={len(out)}, pred={pred.shape[0]}")

    for i, h in enumerate(HORIZONS_HOURS):
        out[f"y_true_hs_{h}h"] = out[f"target_hs_{h}h"].values
        out[f"pred_tcn_hs_{h}h"] = pred[:, i]
        out[f"pred_persistence_hs_{h}h"] = out["persistence_hs"].values

        out[f"true_risk_{h}h"] = risk_level_from_hs(out[f"y_true_hs_{h}h"]).values
        out[f"pred_tcn_risk_{h}h"] = risk_level_from_hs(out[f"pred_tcn_hs_{h}h"]).values
        out[f"pred_persistence_risk_{h}h"] = risk_level_from_hs(out[f"pred_persistence_hs_{h}h"]).values

    out["split"] = split_name

    return out


# Construir test solo después del entrenamiento para no ocupar RAM durante model.fit.
test_ds, test_meta = build_split_sequence(
    "test",
    max_sequences_total=MAX_TEST_SEQUENCES,
    shuffle=False,
    build_metadata=True,
)

if test_ds is None:
    raise ValueError("Test está vacío.")

TEST_SEQUENCE_COUNT = test_ds.n_samples
print("Batches test:", len(test_ds))
print("Secuencias test:", f"{TEST_SEQUENCE_COUNT:,}")

test_meta.to_csv(RESULTS_DIR / "test_sequence_metadata.csv", index=False)

val_predictions = predict_dataset(val_ds, val_meta, "val")
test_predictions = predict_dataset(test_ds, test_meta, "test")

predictions_all = pd.concat([val_predictions, test_predictions], ignore_index=True)

predictions_all.to_parquet(
    RESULTS_DIR / "val_test_predictions.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print("Predicciones:", predictions_all.shape)
display(predictions_all.head())


## Celda 12 — Métricas de regresión TCN

In [ ]:
regression_rows = []

for split_name, pred_df in [
    ("val", val_predictions),
    ("test", test_predictions),
]:
    for h in HORIZONS_HOURS:
        y_true = pred_df[f"y_true_hs_{h}h"]
        pred_tcn = pred_df[f"pred_tcn_hs_{h}h"]
        pred_persistence = pred_df[f"pred_persistence_hs_{h}h"]

        m_tcn = regression_metrics(y_true, pred_tcn)
        m_pers = regression_metrics(y_true, pred_persistence)

        regression_rows.append(
            {
                "horizon_hours": h,
                "target": f"target_hs_{h}h",
                "split": split_name,
                "model": "TCN",
                **m_tcn,
            }
        )

        regression_rows.append(
            {
                "horizon_hours": h,
                "target": f"target_hs_{h}h",
                "split": split_name,
                "model": "Persistence",
                **m_pers,
            }
        )

regression_metrics_df = pd.DataFrame(regression_rows)
regression_metrics_df.to_csv(RESULTS_DIR / "metrics_regression.csv", index=False)

display(regression_metrics_df)

## Celda 13 — Métricas de riesgo derivado TCN

In [ ]:
risk_rows = []

for split_name, pred_df in [
    ("val", val_predictions),
    ("test", test_predictions),
]:
    for h in HORIZONS_HOURS:
        y_true = pred_df[f"true_risk_{h}h"]
        pred_tcn = pred_df[f"pred_tcn_risk_{h}h"]
        pred_persistence = pred_df[f"pred_persistence_risk_{h}h"]

        m_tcn = classification_metrics(y_true, pred_tcn)
        m_pers = classification_metrics(y_true, pred_persistence)

        risk_rows.append(
            {
                "horizon_hours": h,
                "target": f"target_risk_{h}h",
                "split": split_name,
                "model": "TCN_risk_derived",
                **m_tcn,
            }
        )

        risk_rows.append(
            {
                "horizon_hours": h,
                "target": f"target_risk_{h}h",
                "split": split_name,
                "model": "PersistenceRisk",
                **m_pers,
            }
        )

        for model_name, pred_values in [
            ("TCN_risk_derived", pred_tcn),
            ("PersistenceRisk", pred_persistence),
        ]:
            yt = pd.Series(np.asarray(y_true), dtype="Int64")
            yp = pd.Series(np.asarray(pred_values), dtype="Int64")
            mask = yt.notna() & yp.notna()

            cm = confusion_matrix(
                yt[mask].astype(int),
                yp[mask].astype(int),
                labels=RISK_CLASSES,
            )

            pd.DataFrame(
                cm,
                index=[f"true_{i}" for i in RISK_CLASSES],
                columns=[f"pred_{i}" for i in RISK_CLASSES],
            ).to_csv(RESULTS_DIR / f"confusion_matrix_{model_name}_h{h}_{split_name}.csv")

risk_metrics_df = pd.DataFrame(risk_rows)
risk_metrics_df.to_csv(RESULTS_DIR / "metrics_risk.csv", index=False)

display(risk_metrics_df)

## Celda 14 — Comparativa principal TCN vs persistencia

In [ ]:
test_reg = regression_metrics_df[regression_metrics_df["split"] == "test"].copy()
test_risk = risk_metrics_df[risk_metrics_df["split"] == "test"].copy()

display(test_reg)
display(test_risk)

# Mejora frente a persistencia.
improvement_rows = []

for h in HORIZONS_HOURS:
    subset = test_reg[test_reg["horizon_hours"] == h]

    tcn_row = subset[subset["model"] == "TCN"].iloc[0]
    pers_row = subset[subset["model"] == "Persistence"].iloc[0]

    improvement_rows.append(
        {
            "horizon_hours": h,
            "mae_persistence": pers_row["mae"],
            "mae_tcn": tcn_row["mae"],
            "mae_improvement_pct": (pers_row["mae"] - tcn_row["mae"]) / pers_row["mae"] * 100 if pers_row["mae"] > 0 else np.nan,
            "rmse_persistence": pers_row["rmse"],
            "rmse_tcn": tcn_row["rmse"],
            "rmse_improvement_pct": (pers_row["rmse"] - tcn_row["rmse"]) / pers_row["rmse"] * 100 if pers_row["rmse"] > 0 else np.nan,
            "r2_tcn": tcn_row["r2"],
            "corr_tcn": tcn_row["corr"],
        }
    )

improvement_df = pd.DataFrame(improvement_rows)
improvement_df.to_csv(RESULTS_DIR / "test_improvement_vs_persistence.csv", index=False)

display(improvement_df)

## Celda 15 — Gráficos de métricas TCN

In [ ]:
plt.figure(figsize=(9, 5))

for model_name in ["Persistence", "TCN"]:
    data = test_reg[test_reg["model"] == model_name].sort_values("horizon_hours")
    plt.plot(data["horizon_hours"], data["mae"], marker="o", label=model_name)

plt.xlabel("Horizonte predictivo (h)")
plt.ylabel("MAE hs (m)")
plt.title("TCN vs persistencia — MAE en test")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot_test_mae_by_horizon.png", dpi=160)
plt.show()

plt.figure(figsize=(9, 5))

for model_name in ["Persistence", "TCN"]:
    data = test_reg[test_reg["model"] == model_name].sort_values("horizon_hours")
    plt.plot(data["horizon_hours"], data["rmse"], marker="o", label=model_name)

plt.xlabel("Horizonte predictivo (h)")
plt.ylabel("RMSE hs (m)")
plt.title("TCN vs persistencia — RMSE en test")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot_test_rmse_by_horizon.png", dpi=160)
plt.show()

plt.figure(figsize=(9, 5))

for model_name in ["PersistenceRisk", "TCN_risk_derived"]:
    data = test_risk[test_risk["model"] == model_name].sort_values("horizon_hours")
    plt.plot(data["horizon_hours"], data["macro_f1"], marker="o", label=model_name)

plt.xlabel("Horizonte predictivo (h)")
plt.ylabel("Macro F1")
plt.title("TCN riesgo derivado vs persistencia — Macro F1")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot_test_macro_f1_risk.png", dpi=160)
plt.show()

plt.figure(figsize=(9, 5))

for model_name in ["PersistenceRisk", "TCN_risk_derived"]:
    data = test_risk[test_risk["model"] == model_name].sort_values("horizon_hours")
    plt.plot(data["horizon_hours"], data["recall_class_3"], marker="o", label=model_name)

plt.xlabel("Horizonte predictivo (h)")
plt.ylabel("Recall clase 3")
plt.title("TCN riesgo derivado — Recall de riesgo extremo")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot_test_recall_class_3.png", dpi=160)
plt.show()

## Celda 16 — Comparación opcional contra LightGBM y XGBoost

In [ ]:
COMPARISON_DIR = GOLD_DIR / "model_results" / "model_comparison"

comparison_available = COMPARISON_DIR.exists()

if comparison_available:
    lgbm_reg_path = COMPARISON_DIR / "lgbm_regression_metrics_recalculated.csv"
    risk_comp_path = COMPARISON_DIR / "comparison_risk_metrics.csv"

    if lgbm_reg_path.exists():
        lgbm_reg = pd.read_csv(lgbm_reg_path)
        lgbm_reg_test = lgbm_reg[lgbm_reg["split"] == "test"].copy()

        combined_reg = pd.concat(
            [
                lgbm_reg_test[lgbm_reg_test["model"].isin(["LightGBMRegressor", "Persistence"])],
                test_reg[test_reg["model"] == "TCN"],
            ],
            ignore_index=True,
        )

        combined_reg.to_csv(RESULTS_DIR / "comparison_regression_tcn_lightgbm.csv", index=False)

        display(combined_reg)

        plt.figure(figsize=(9, 5))
        for model_name in ["Persistence", "LightGBMRegressor", "TCN"]:
            data = combined_reg[combined_reg["model"] == model_name].sort_values("horizon_hours")
            plt.plot(data["horizon_hours"], data["mae"], marker="o", label=model_name)

        plt.xlabel("Horizonte predictivo (h)")
        plt.ylabel("MAE hs (m)")
        plt.title("Comparación regresión hs: Persistencia vs LightGBM vs TCN")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "plot_compare_mae_lightgbm_tcn.png", dpi=160)
        plt.show()

    if risk_comp_path.exists():
        risk_comp = pd.read_csv(risk_comp_path)
        risk_comp_test = risk_comp[risk_comp["split"] == "test"].copy()

        combined_risk = pd.concat(
            [
                risk_comp_test,
                test_risk[test_risk["model"] == "TCN_risk_derived"],
            ],
            ignore_index=True,
        )

        combined_risk.to_csv(RESULTS_DIR / "comparison_risk_tcn_lightgbm_xgb.csv", index=False)

        display(combined_risk)

        plt.figure(figsize=(9, 5))
        for model_name in [
            "PersistenceRisk",
            "LightGBMRegressor_risk_derived",
            "XGBClassifier_direct_risk",
            "TCN_risk_derived",
        ]:
            data = combined_risk[combined_risk["model"] == model_name].sort_values("horizon_hours")
            if data.empty:
                continue
            plt.plot(data["horizon_hours"], data["macro_f1"], marker="o", label=model_name)

        plt.xlabel("Horizonte predictivo (h)")
        plt.ylabel("Macro F1")
        plt.title("Comparación riesgo: Persistencia vs LightGBM vs XGBoost vs TCN")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "plot_compare_risk_macro_f1_all_models.png", dpi=160)
        plt.show()

else:
    print("No existe model_comparison/. Se omite comparación externa.")

## Celda 17 — Guardar configuración del experimento

In [ ]:
training_config = {
    "model_family": "TCN",
    "task": "multi_horizon_wave_height_regression",
    "risk_derivation": "thresholds_from_predicted_hs",
    "execution_environment": {
        "platform": platform.platform(),
        "machine": platform.machine(),
        "python": sys.version,
        "tensorflow": tf.__version__,
        "gpu_devices": [str(x) for x in tf.config.list_physical_devices("GPU")],
        "cpu_cores": CPU_CORES,
        "tf_intra_threads": os.environ.get("TF_NUM_INTRAOP_THREADS"),
        "tf_inter_threads": os.environ.get("TF_NUM_INTEROP_THREADS"),
        "ram_total_gb": RAM_TOTAL_GB,
        "ram_available_at_start_gb": RAM_AVAILABLE_GB,
    },
    "train_profile": TRAIN_PROFILE,
    "base_dir": str(BASE_DIR),
    "horizons_hours": HORIZONS_HOURS,
    "target_cols": TARGET_COLS,
    "sequence_length": SEQUENCE_LENGTH,
    "batch_size": BATCH_SIZE,
    "epochs_requested": EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "huber_delta": HUBER_DELTA,
    "tcn_filters": TCN_FILTERS,
    "tcn_dilations": TCN_DILATIONS,
    "tcn_dropout": TCN_DROPOUT,
    "dense_units_1": DENSE_UNITS_1,
    "dense_units_2": DENSE_UNITS_2,
    "features_count": len(feature_cols),
    "feature_cols": feature_cols,
    "categorical_features": categorical_features,
    "encoded_categorical_features": encoded_categorical_features,
    "use_common_zones_only": USE_COMMON_ZONES_ONLY,
    "common_zones_count": len(common_zones),
    "common_zones": sorted(list(common_zones)),
    "max_train_sequences": MAX_TRAIN_SEQUENCES,
    "max_val_sequences": MAX_VAL_SEQUENCES,
    "max_test_sequences": MAX_TEST_SEQUENCES,
    "sequence_counts": {
        "train": int(globals().get("TRAIN_SEQUENCE_COUNT", -1)),
        "val": int(globals().get("VAL_SEQUENCE_COUNT", -1)),
        "test": int(globals().get("TEST_SEQUENCE_COUNT", -1)),
    },
    "risk_thresholds": RISK_THRESHOLDS,
    "model_paths": {
        "best": str(MODELS_DIR / "tcn_best.keras"),
        "final": str(MODELS_DIR / "tcn_final.keras"),
    },
}

save_json(training_config, RESULTS_DIR / "training_config.json")
save_json(training_config, MODELS_DIR / "training_config.json")

print("Configuración guardada.")
print(RESULTS_DIR / "training_config.json")
print(MODELS_DIR / "training_config.json")


## Celda 18 — Validación final

In [ ]:
required_files = [
    MODELS_DIR / "tcn_best.keras",
    MODELS_DIR / "tcn_final.keras",
    RESULTS_DIR / "training_history.csv",
    RESULTS_DIR / "metrics_regression.csv",
    RESULTS_DIR / "metrics_risk.csv",
    RESULTS_DIR / "val_test_predictions.parquet",
    RESULTS_DIR / "test_improvement_vs_persistence.csv",
    RESULTS_DIR / "training_config.json",
    RESULTS_DIR / "tcn_feature_scaler_metadata.json",
]

missing = [str(p) for p in required_files if not Path(p).exists()]

if missing:
    raise FileNotFoundError("Faltan archivos de salida: " + json.dumps(missing, indent=2))

test_tcn = regression_metrics_df[
    (regression_metrics_df["split"] == "test")
    & (regression_metrics_df["model"] == "TCN")
]

if test_tcn["horizon_hours"].nunique() != len(HORIZONS_HOURS):
    raise ValueError("No hay métricas test TCN para todos los horizontes.")

if test_tcn["mae"].isna().any() or test_tcn["rmse"].isna().any():
    raise ValueError("Hay métricas MAE/RMSE nulas en test.")

print("Archivos generados en", RESULTS_DIR)
for p in sorted(RESULTS_DIR.glob("*")):
    print("-", p)

print("\nModelos generados en", MODELS_DIR)
for p in sorted(MODELS_DIR.glob("*")):
    print("-", p)

print("\nResumen test TCN:")
display(test_tcn)

print("\n✅ Entrenamiento TCN completado correctamente.")

## Resultado esperado

Al final debe aparecer:

```text
✅ Entrenamiento TCN completado correctamente.
```

Archivos principales:

```text
models/tcn/tcn_best.keras
models/tcn/tcn_final.keras

gold/model_results/tcn/metrics_regression.csv
gold/model_results/tcn/metrics_risk.csv
gold/model_results/tcn/val_test_predictions.parquet
gold/model_results/tcn/test_improvement_vs_persistence.csv
gold/model_results/tcn/training_history.csv
gold/model_results/tcn/comparison_regression_tcn_lightgbm.csv
gold/model_results/tcn/comparison_risk_tcn_lightgbm_xgb.csv
```

Interpretación:

```text
- Si TCN supera a LightGBM en MAE/RMSE, será candidato a modelo final de hs.
- Si no lo supera, sigue siendo útil como comparación deep learning.
- Si mejora recall extremo, puede ser candidato para un ensemble.
```

Siguiente paso:

```text
13B_model_comparison_tcn_classical.ipynb
```

o directamente actualizar el informe final con los resultados TCN.